In [ ]:
# 忽视警告，这个库是内置的，不需要安装
from pathlib import Path
import importlib.util
import sys
import warnings

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

support_path_candidates = [
    Path("cuda_training_support.py"),
    Path("..") / "cuda_training_support.py",
]
SUPPORT_PATH = next((path.resolve() for path in support_path_candidates if path.exists()), None)
if SUPPORT_PATH is None:
    raise FileNotFoundError("找不到 cuda_training_support.py")

support_spec = importlib.util.spec_from_file_location("cuda_training_support", SUPPORT_PATH)
cuda_training_support = importlib.util.module_from_spec(support_spec)
sys.modules["cuda_training_support"] = cuda_training_support
support_spec.loader.exec_module(cuda_training_support)

NOTEBOOK_CONFIG = cuda_training_support.build_notebook_run_config()
DEFAULT_TARGET_COLUMN = cuda_training_support.DEFAULT_TARGET_COLUMN

# 兼容从项目根目录或 Notebook 所在目录启动内核的两种情况。
DATA_PATH = cuda_training_support.resolve_data_path(start_dir=Path.cwd())

print(
    cuda_training_support.format_notebook_run_summary(
        NOTEBOOK_CONFIG,
        data_path=DATA_PATH,
    )
)


In [3]:
import pandas as pd

random_seed = NOTEBOOK_CONFIG.random_seed

data = cuda_training_support.load_training_dataframe(
    data_path=DATA_PATH,
    random_seed=random_seed,
    run_mode=NOTEBOOK_CONFIG.run_mode,
    sample_size=NOTEBOOK_CONFIG.sample_size,
    target_column=DEFAULT_TARGET_COLUMN,
)

# 显示前几行数据，以确认数据已正确加载
(data.head(10))


,毒性,PN,ID,BP,BPP,MaxM,MaxMP,MinM,MM,MSD,IM,ISD,RETENTION_TIME,COLLISION_ENERGY,PRECURSOR_TYPE
0,0,4,249.750000,182.128900,16.03140,195.16050,13.031600,72.044400,153.857825,48.343590,3.329761e+06,3.439429e+06,5.0,35.0,1.0
1,0,3,333.000000,83.050300,2.01570,101.06080,18.010500,81.034600,88.381900,9.003023,2.051420e+04,7.277040e+03,9.798,45.0,0.0
2,1,25,39.960000,63.005000,0.03500,153.98260,25.957200,42.043900,92.923700,27.992386,3.056000e+00,7.479630e+00,7.429,60.0,1.0
3,0,25,39.960000,77.044500,1.01030,115.04350,1.009000,38.018600,79.188564,20.085917,2.040000e+00,3.322168e+00,5.203,130.0,1.0
4,1,4,249.750000,395.300000,18.10000,542.40000,147.100000,365.300000,420.050000,71.441882,2.849013e+06,2.568194e+06,NaN,20.0,0.0
5,0,1,999.000000,122.073600,0.00000,122.07360,0.000000,122.073600,122.073600,0.000000,9.678258e+05,0.000000e+00,2.215,55.0,1.0
6,1,66,15.136364,444.376000,14.01800,537.44200,1.004000,95.086000,242.237379,110.458475,5.717273e+04,5.913864e+04,NaN,11.0,1.0
7,1,2,499.500000,160.050600,0.00000,192.07680,32.026200,160.050600,176.063700,16.013100,3.242676e+06,1.048466e+06,1.107,40.0,1.0
8,0,9,111.000000,328.158970,0.01169,328.15897,0.011690,162.059920,268.653232,54.547444,2.707778e+02,2.648621e+02,3.043967,6.0,0.0
9,1,3,333.000000,41.014522,0.00000,83.03632,17.026549,41.014522,63.353538,17.257840,3.994961e+01,4.271762e+01,NaN,10.0,0.0


In [4]:
import pandas as pd
data['RETENTION_TIME'] = pd.to_numeric(data['RETENTION_TIME'], errors='coerce').astype('float64')
# 假设 df 是你的 DataFrame
print("转换前 RETENTION_TIME 的示例值：")
print(data['RETENTION_TIME'].head())

# 检查非数值数据（如字符串、空值等）
non_numeric = pd.to_numeric(data['RETENTION_TIME'], errors='coerce').isna()
print("\n非数值数据数量：", non_numeric.sum())
print("非数值数据示例：")
print(data[non_numeric]['RETENTION_TIME'].unique())  # 查看具体非数值内容

转换前 RETENTION_TIME 的示例值：
0    5.000
1    9.798
2    7.429
3    5.203
4      NaN
Name: RETENTION_TIME, dtype: float64

非数值数据数量： 193
非数值数据示例：
[nan]


In [5]:
print("样本总数:", len(data))
print("标签分布:")
data[DEFAULT_TARGET_COLUMN].value_counts()


样本总数: 1024
标签分布:


毒性
0    665
1    359
Name: count, dtype: int64

In [6]:
prepared = cuda_training_support.prepare_lightgbm_training_data(
    data,
    target_column=DEFAULT_TARGET_COLUMN,
    random_state=NOTEBOOK_CONFIG.random_seed,
    test_size=NOTEBOOK_CONFIG.test_size,
)

X_train = prepared["X_train"]
X_test = prepared["X_test"]
y_train = prepared["y_train"]
y_test = prepared["y_test"]
scaler = prepared["scaler"]

print("过采样方法:", "notebook_smote")
print("过采样后的训练集形状:", X_train.shape)
print("测试集形状:", X_test.shape)
print("过采样后的标签分布:", pd.Series(y_train).value_counts())


过采样方法: notebook_smote
过采样后的训练集形状: (1080, 14)
测试集形状: (205, 14)
过采样后的标签分布: 毒性
1    540
0    540
Name: count, dtype: int64


## LightGBM（CUDA-only）

- 训练设备固定为 `cuda`，禁止 CPU fallback。
- `NOTEBOOK_CONFIG` 由 `build_notebook_run_config()` 生成，启动内核前可通过这些环境变量覆盖：`LGBM_NOTEBOOK_RUN_MODE`、`LGBM_SMOKE_SAMPLE_SIZE`、`LGBM_BAYES_N_ITER`、`LGBM_CV_FOLDS`、`LGBM_RANDOM_SEED`、`LGBM_TEST_SIZE`、`LGBM_MODEL_N_JOBS`、`LGBM_SEARCH_N_JOBS`。
- Notebook 默认使用 `smoke` 模式做轻量验证；设置 `LGBM_NOTEBOOK_RUN_MODE=full` 可切换为全量训练。
- 训练前会执行一次 CUDA 预检；如果当前 `lightgbm` 不是带 CUDA 的构建，会直接报错并停止。
- 官方当前不支持 Windows 上的 CUDA 版 LightGBM。需要把训练移动到 Linux 或 WSL2，并先执行：

```bash
pip uninstall -y lightgbm
pip install lightgbm --no-binary lightgbm --config-settings=cmake.define.USE_CUDA=ON
```


In [ ]:
import sys

# 如果内核从项目根目录启动，先移除本地 lightgbm 目录对官方包导入的遮蔽。
current_dir = Path.cwd().resolve()
if (current_dir / "lightgbm").is_dir():
    sys.path = [
        path
        for path in sys.path
        if Path(path or current_dir).resolve() != current_dir
    ]

import lightgbm as lgb
from skopt import BayesSearchCV

lgbm_version = cuda_training_support.validate_lightgbm_cuda_build(
    random_state=NOTEBOOK_CONFIG.random_seed,
)
print("LightGBM CUDA preflight:", lgbm_version)

# 定义 LightGBM 模型，设备被强制锁定为 CUDA。
lgb_model = cuda_training_support.build_lgbm_classifier(
    random_state=NOTEBOOK_CONFIG.random_seed,
    model_n_jobs=NOTEBOOK_CONFIG.model_n_jobs,
)

# 定义贝叶斯搜索空间。smoke 模式使用更小的搜索范围，避免误触全量计算。
search_spaces = cuda_training_support.get_lgbm_search_spaces(NOTEBOOK_CONFIG.run_mode)

# 初始化贝叶斯搜索
bayes_search = BayesSearchCV(
    estimator=lgb_model,
    search_spaces=search_spaces,
    n_iter=NOTEBOOK_CONFIG.bayes_n_iter,
    cv=NOTEBOOK_CONFIG.cv_folds,
    scoring="roc_auc",
    n_jobs=NOTEBOOK_CONFIG.search_n_jobs,
    verbose=1,
    random_state=NOTEBOOK_CONFIG.random_seed,
)

# 在训练集上执行贝叶斯搜索
bayes_search.fit(X_train, y_train)

# 输出最佳参数
print("最佳参数组合:", bayes_search.best_params_)
print("最佳验证集AUC:", bayes_search.best_score_)

# 使用最佳模型预测测试集
y_pred_proba = bayes_search.best_estimator_.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)
print("测试集AUC:", test_auc)

# 可选：也输出测试集的准确率和 F1 分数
y_pred = bayes_search.best_estimator_.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)
print("测试集准确率:", test_accuracy)
print("测试集F1分数:", test_f1)


In [ ]:
# 使用最佳参数训练模型
best_lgb = bayes_search.best_estimator_

# ============ 训练集评估 ============
y_train_pred = best_lgb.predict(X_train)
y_train_proba = best_lgb.predict_proba(X_train)[:, 1]

# 计算训练集指标
train_metrics = {
    'AUC': roc_auc_score(y_train, y_train_proba),
    'Accuracy': accuracy_score(y_train, y_train_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_train, y_train_pred),
    'Precision': precision_score(y_train, y_train_pred),
    'Recall': recall_score(y_train, y_train_pred),
    'F1': f1_score(y_train, y_train_pred)
}

# 计算Specificity
tn, fp, fn, tp = confusion_matrix(y_train, y_train_pred).ravel()
train_metrics['Specificity'] = tn / (tn + fp)

# ============ 测试集评估 ============
y_test_pred = best_lgb.predict(X_test)
y_test_proba = best_lgb.predict_proba(X_test)[:, 1]

# 计算测试集指标
test_metrics = {
    'AUC': roc_auc_score(y_test, y_test_proba),
    'Accuracy': accuracy_score(y_test, y_test_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_test_pred),
    'Precision': precision_score(y_test, y_test_pred),
    'Recall': recall_score(y_test, y_test_pred),
    'F1': f1_score(y_test, y_test_pred)
}

# 计算Specificity
tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
test_metrics['Specificity'] = tn / (tn + fp)

# ============ 打印结果 ============
print("\n=== 训练集性能 ===")
for metric, value in train_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

print("\n=== 测试集性能 ===")
for metric, value in test_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

# 打印混淆矩阵
print("\n测试集混淆矩阵:")
print(confusion_matrix(y_test, y_test_pred))

# 特征重要性可视化（可选）
lgb.plot_importance(best_lgb, max_num_features=20)
plt.tight_layout()
plt.show()